# 00b · Mirror gated models

Downloads the gated Hugging Face models once into this notebook's output, so notebooks pushed with the Kaggle CLI (which get no secrets) can load them. **Run this one from the Kaggle editor with *Save Version*, `HF_TOKEN` ticked.** Re-run only to change the model list.

| | |
|---|---|
| **Kaggle settings** | Accelerator: None (CPU is enough) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) |
| **Reads** | Hugging Face Hub (gated models) |
| **Writes** (`/kaggle/working/`) | `models/<org>__<name>/`, `models/mirror_manifest.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [ ]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

In [ ]:
from huggingface_hub import HfApi, snapshot_download

# Accept each licence on its Hub page first. A model that fails is reported and skipped.
MODELS = ["CohereLabs/tiny-aya-global", "meta-llama/Llama-Guard-3-1B"]
PATTERNS = ["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja", "tokenizer*"]
OUT = sp.out_dir("models")
sp.hf_login()

In [ ]:
manifest = {}
for name in MODELS:
    local = OUT / name.replace("/", "__")
    try:
        sha = HfApi().model_info(name).sha
        snapshot_download(name, revision=sha, local_dir=local, allow_patterns=PATTERNS)
        size = sum(f.stat().st_size for f in local.rglob("*") if f.is_file()) / 1e9
        manifest[name] = {"status": "ok", "revision": sha, "gb": round(size, 2)}
    except Exception as e:
        manifest[name] = {"status": f"failed: {type(e).__name__}: {str(e)[:200]}"}
    print(name, manifest[name])
sp.save_json({"models": manifest, "provenance": sp.provenance()}, OUT / "mirror_manifest.json")

Later notebooks find a mirrored model automatically (`sp.model_source`) when this notebook's output is
attached. Keep the notebook **private**: the mirror is for this project's own use under each model's licence.